# EXP-2026-009 — Q5-D negative-control null 아티팩트 복구 (미실행 템플릿)

이 노트북은 **포장 복구**다. 과학 실험이 아니다.

- beat join 재실행 없음 · null 재실행 없음 · `J` 값 계산 없음
- 기존 Drive bundle·shard **수정·삭제·덮어쓰기 없음**
- 새 corrective 폴더 하나만 만들고, 거기에 **`BUNDLE_FILES` 12개 외에는 아무것도 넣지 않는다**

명세: `experiments/specs/EXP-2026-009-q5d-null-artifact-repair.md`
판정 출처: `EXP-2026-008` 실행계약 Decision log의 `P2_PRODUCER_ARTIFACT_OMISSION`

**현재 실행 승인은 없다.** 모듈의 `EXECUTION_APPROVAL_RECORD['granted']` 가
`False` 이므로 아래 실행 셀은 `REPAIR_NOT_APPROVED` 로 거부된다. 승인은 별도
결정이고 별도 PR이 연다 — 이 노트북은 그것을 우회하지 않는다.

In [ ]:
# 1. ENVIRONMENT — 저장소를 찾아 로드한다. "경로가 존재한다"가 아니라
#    "필요한 모듈이 거기 있다"로 판정한다(quest56 과 같은 규칙).
import os, sys, json, subprocess

REPO = ''          # 경로를 직접 알면 절대경로를 적는다 (그러면 탐색 생략)
REPO_URL = 'https://github.com/ehdbddl06001-ui/my-github-test.git'
REPO_BRANCH = 'main'
CLONE_TO = '/content/repo'
NEEDED = ('q5d_order_preserving_beat_join.py',
          'q5d_null_artifact_repair.py')


def _is_repo(path):
    if not path:
        return False
    here = os.path.join(path, 'mit-bih')
    return all(os.path.isfile(os.path.join(here, n)) for n in NEEDED)


def _candidates():
    if REPO:
        yield REPO
    yield '/content/repo'
    yield '/content/my-github-test'
    walk = os.path.abspath(os.getcwd())
    while True:
        yield walk
        parent = os.path.dirname(walk)
        if parent == walk:
            break
        walk = parent
    for base in (os.path.abspath(os.getcwd()), '/content',
                 '/content/drive/MyDrive'):
        try:
            for name in sorted(os.listdir(base)):
                yield os.path.join(base, name)
        except OSError:
            pass


FOUND = next((c for c in _candidates() if _is_repo(c)), '')

if not FOUND:
    print('저장소를 찾지 못했다. clone 을 시도한다:', REPO_URL, REPO_BRANCH)
    _r = subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1',
         REPO_URL, CLONE_TO], capture_output=True, text=True)
    print((_r.stdout or '').strip() or (_r.stderr or '').strip())
    if _is_repo(CLONE_TO):
        FOUND = CLONE_TO

if not FOUND:
    raise RuntimeError(
        '저장소를 찾지 못했다. 다음 중 하나를 하고 이 셀을 다시 실행하라.\n'
        f'  1) 직접 클론:  !git clone --branch {REPO_BRANCH} {REPO_URL} '
        f'{CLONE_TO}\n'
        '     (토큰을 이 노트북 셀에 적지 마라 — 출력에 저장된다.)\n'
        '  2) Drive 에 사본이 있으면 마운트한 뒤 위 REPO 에 절대경로를 적는다.\n'
        f'  판정 기준: <REPO>/mit-bih/ 안에 {", ".join(NEEDED)} 가 모두 있어야 한다.')

REPO = FOUND
_MITBIH = os.path.join(REPO, 'mit-bih')
if _MITBIH not in sys.path:
    sys.path.insert(0, _MITBIH)

import q5d_order_preserving_beat_join as BJ
import q5d_null_artifact_repair as R

MISSING = [n for n in R.module_capabilities() if not hasattr(R, n)]
assert not MISSING, f'stale repair clone, missing {MISSING}'

print('REPO         :', REPO)
print('repair module:', R.__file__)
print('frozen Q5-D  :', BJ.__file__)
print()
print(R.design_card())

In [ ]:
# 2. 동결 모듈 신원 — shard 폴더 이름에 박힌 코드 해시와 같은 값이어야 한다.
#    다르면 지금 읽으려는 코드가 그 shard 를 만든 코드가 아니라는 뜻이다.
print('registered :', R.FROZEN_Q5D_SHA256)
print('imported   :', R.frozen_q5d_sha256())
print('match      :', R.frozen_q5d_sha256() == R.FROZEN_Q5D_SHA256)
print()
print('contract   :', len(R.BUNDLE_FILES), '개 —', list(R.BUNDLE_FILES))
print('source     :', len(R.SOURCE_BUNDLE_FILES), '개 (12 − 재구성 대상 1)')
print('재구성 대상 :', R.MISSING_ARTIFACT)
print('NPZ 배열   :', list(R.NPZ_ARRAYS), '· 각 float64', f'({R.N_REPLICATES},)')

In [ ]:
# 3. 합성 fixture 검증 — 실제 자산을 열기 전에 통과해야 하는 가장 싼 관문.
#    실패하면 stderr 와 종료 코드를 보이고 여기서 멈춘다(조용한 실패 금지).
import subprocess, sys

_res = subprocess.run(
    [sys.executable, os.path.join(REPO, 'mit-bih',
                                  'test_q5d_null_artifact_repair.py')],
    capture_output=True, text=True)
print((_res.stdout or '').strip() or '(stdout 없음)')
if _res.returncode != 0:
    print('--- stderr ---')
    print((_res.stderr or '').strip() or '(stderr 없음)')
    raise RuntimeError(
        f'합성 fixture 가 실패했다 (exit {_res.returncode}). '
        f'셀 5(실행 셀)를 누르지 마라 — 실패한 코드로 실제 자산을 열지 않는다.')

In [ ]:
# 4. 경로 — Drive 마운트 뒤 실제 위치를 적는다. 아직 아무것도 열지 않는다.
#    · SHARD_DIR   : 기존 100개 null shard 폴더 (읽기 전용)
#    · SOURCE_DIR  : 기존 canonical Q5-D bundle 11개 파일 (읽기 전용)
#    · TARGET_DIR  : 새 corrective 폴더 — 반드시 아직 없는 이름이어야 한다
#
#    TARGET_DIR 을 기존 run 폴더 안이나 그 이름으로 두지 마라. 이 복구는
#    기존 폴더를 절대 건드리지 않는다.
DRIVE_ROOT = '/content/drive/MyDrive/MedKOS/ecg-model'

SHARD_DIR  = ''   # 예: f'{DRIVE_ROOT}/runs/EXP-2026-007_q5d_beat_join_null_shards_DS1_6b098c67df3c'
SOURCE_DIR = ''   # 예: f'{DRIVE_ROOT}/runs/20260811T035108_EXP-2026-007_q5d_beat_join_DS1_GATE'
TARGET_DIR = ''   # 예: f'{DRIVE_ROOT}/runs/<ts>_EXP-2026-009_q5d_null_artifact_repair_corrective'

for _label, _path in (('SHARD_DIR', SHARD_DIR), ('SOURCE_DIR', SOURCE_DIR),
                      ('TARGET_DIR', TARGET_DIR)):
    print(f'{_label:11s}:', _path or '(미지정)')

print()
print('실행 승인 :', bool(R.EXECUTION_APPROVAL_RECORD.get('granted')))
print(R.APPROVAL_NOTE)

In [ ]:
# 5. 실행 — 승인 전에는 REPAIR_NOT_APPROVED 로 거부된다. 그것이 정상 동작이다.
#    승인이 나면 이 셀이 그대로 route 를 돈다: 자격검증 → 재구성 →
#    null_summary 대조 → NPZ 계약 → corrective 폴더 조립 → 재검증.
#    하나라도 실패하면 폴더는 만들어지지 않는다.
APPROVAL = R.EXECUTION_APPROVAL_TOKEN   # 리터럴을 적지 않는다

try:
    DECISION = R.run_repair(SHARD_DIR, SOURCE_DIR, TARGET_DIR, APPROVAL)
    print('status:', DECISION['status'])
except R.RepairError as _error:
    DECISION = None
    print('STOP:', _error.reason)
    print(_error)

In [ ]:
# 6. 보고 — 이 셀의 저장된 출력이 외부 기록이다(corrective 폴더 안에는
#    provenance 파일을 넣지 않는다). 여기 나온 digest 를 별도 PR 이
#    Decision log · ASSETS.md · PROJECT_STATE.md 로 옮긴다.
if DECISION is None:
    print('실행되지 않았다 — 위 셀의 STOP 사유를 보라. 기록할 값이 없다.')
else:
    print(R.report_markdown(DECISION))
    print()
    print('--- 복사해 갈 값 ---')
    print('NPZ SHA-256      :', DECISION['npz']['sha256'])
    print('corrective 폴더  :', DECISION['corrective_bundle']['directory'])
    print('파일 수          :', len(DECISION['corrective_bundle']['listing']))
    print()
    print(json.dumps(DECISION['verification']['observed'], indent=2,
                     sort_keys=True))

## 이 노트북이 하지 않는 것

`detect_r()` · beat join 재실행 · null 재실행 · M0~M4 집계 · DS2 per-beat label ·
V10 probability · association · S PR-AUC · 학습 · 기존 Drive 파일 이동·삭제·
덮어쓰기 · frozen 모듈 수정 · 12파일 계약 완화 · 값 등록.

**등록은 이 실행의 결과가 아니다.** corrective 폴더가 만들어져도 folder id·
lineage·NPZ digest 는 별도 등록 PR 로만 들어가고, Q5-E PREP P1/P2 재실행은
그와 또 별개의 사용자 승인을 받는다.